# Team Operations : Reset, Stop, Resume and Abort

In [16]:
import asyncio
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
model_client = OpenAIChatCompletionClient(model='gpt-4o', api_key=api_key)

In [17]:
from autogen_agentchat.agents import AssistantAgent
add_1_agent_first = AssistantAgent(
    name = 'add_1_agent_first',
    model_client=model_client,
    system_message="Add 1 to the number, first number is 0. Give result as output"
)

add_1_agent_second = AssistantAgent(
    name = 'add_1_agent_second',
    model_client=model_client,
    system_message="Add 1 to the number. Give result as output."
)
 
add_1_agent_third = AssistantAgent(
    name = 'add_1_agent_third',
    model_client=model_client,
    system_message="Add 1 to the number. Give result as output."
)


In [18]:
from autogen_agentchat.teams import RoundRobinGroupChat

team = RoundRobinGroupChat(
    [add_1_agent_first, add_1_agent_second, add_1_agent_third],
    max_turns=3
)

In [19]:
from autogen_agentchat.ui import Console

await Console(team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
1
---------- TextMessage (add_1_agent_second) ----------
2
---------- TextMessage (add_1_agent_third) ----------
3


TaskResult(messages=[TextMessage(id='5d8b9e3a-33ec-4b1b-b4bf-ae1f0c69381e', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=24, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 1, 31, 6, 47, 9, 569470, tzinfo=datetime.timezone.utc), content='1', type='TextMessage'), TextMessage(id='ece24061-db24-4b1d-89cb-a724b5d57925', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=29, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 1, 31, 6, 47, 10, 62750, tzinfo=datetime.timezone.utc), content='2', type='TextMessage'), TextMessage(id='673ea83f-f079-4b5f-90cb-e1ef6078d4bc', source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=39, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 1, 31, 6, 47, 10, 566538, tzinfo=datetime.timezone.utc), content='3', type='TextMessage')], stop_reason='Maximum number of turns 3 reached.')

# Resuming a Team
Teams are stateful and maintains the conversation history and context after each run, unless you reset the team.


We can resume a team to continue from where it left off by calling the run() or run_stream() method without a **new task**


In [20]:
await Console(team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
4
---------- TextMessage (add_1_agent_second) ----------
5
---------- TextMessage (add_1_agent_third) ----------
6


TaskResult(messages=[TextMessage(id='eff34b6b-4431-4f80-a49c-f94279e8a3cd', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=50, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 1, 31, 6, 47, 11, 188483, tzinfo=datetime.timezone.utc), content='4', type='TextMessage'), TextMessage(id='2c1df272-80df-4f48-bf63-03a629fb2b32', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=55, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 1, 31, 6, 47, 12, 824108, tzinfo=datetime.timezone.utc), content='5', type='TextMessage'), TextMessage(id='05cb257e-7eca-4c30-82cf-0984058351f2', source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=64, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 1, 31, 6, 47, 13, 352609, tzinfo=datetime.timezone.utc), content='6', type='TextMessage')], stop_reason='Maximum number of turns 3 reached.')

In [21]:
await Console(team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
7
---------- TextMessage (add_1_agent_second) ----------
8
---------- TextMessage (add_1_agent_third) ----------
9


TaskResult(messages=[TextMessage(id='6708cfbe-988d-460c-8e27-a5393f97772d', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=76, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 1, 31, 6, 47, 13, 951310, tzinfo=datetime.timezone.utc), content='7', type='TextMessage'), TextMessage(id='d593fe8d-f44f-4551-8401-1a5844a75c5e', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=81, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 1, 31, 6, 47, 14, 669474, tzinfo=datetime.timezone.utc), content='8', type='TextMessage'), TextMessage(id='7b850405-7006-460e-8bfb-810ece6fe220', source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=89, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 1, 31, 6, 47, 15, 384036, tzinfo=datetime.timezone.utc), content='9', type='TextMessage')], stop_reason='Maximum number of turns 3 reached.')

# team resumed from where it left off in the output above, and the first message is from the next agent after the last agent that spoke before the team stopped.

In [22]:
await Console(team.run_stream(task = 'What was the largest number you got in the result?'))

---------- TextMessage (user) ----------
What was the largest number you got in the result?


---------- TextMessage (add_1_agent_first) ----------
The largest number I got in the result sequence was 9.
---------- TextMessage (add_1_agent_second) ----------
The largest number I got was 8.
---------- TextMessage (add_1_agent_third) ----------
The largest number in the overall sequence of results was 9.


TaskResult(messages=[TextMessage(id='5eeb92ba-4d5b-47ab-b68e-f03281610980', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 1, 31, 6, 47, 15, 406347, tzinfo=datetime.timezone.utc), content='What was the largest number you got in the result?', type='TextMessage'), TextMessage(id='231a9a2b-8128-4fe5-9dd1-ee32c514b2b8', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=118, completion_tokens=13), metadata={}, created_at=datetime.datetime(2026, 1, 31, 6, 47, 16, 206047, tzinfo=datetime.timezone.utc), content='The largest number I got in the result sequence was 9.', type='TextMessage'), TextMessage(id='f8f14d8a-dd08-47b7-96de-6f935e50ffd0', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=135, completion_tokens=9), metadata={}, created_at=datetime.datetime(2026, 1, 31, 6, 47, 16, 815720, tzinfo=datetime.timezone.utc), content='The largest number I got was 8.', type='TextMessage'), TextMessage(id='9073b8ef-210f-423e-850d

# Reset our Team

In [23]:
await team.reset() # on_reset() on all agents

In [24]:
await Console(team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
1
---------- TextMessage (add_1_agent_second) ----------
2
---------- TextMessage (add_1_agent_third) ----------
3


TaskResult(messages=[TextMessage(id='240b6749-8b3f-4332-aa01-873fd634362f', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=24, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 1, 31, 6, 47, 18, 168421, tzinfo=datetime.timezone.utc), content='1', type='TextMessage'), TextMessage(id='28640236-99cf-4854-8aae-b90979d41a2c', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=29, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 1, 31, 6, 47, 18, 642136, tzinfo=datetime.timezone.utc), content='2', type='TextMessage'), TextMessage(id='c6d6f1ac-ee28-498f-8d3d-25e00c255a7c', source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=39, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 1, 31, 6, 47, 19, 81011, tzinfo=datetime.timezone.utc), content='3', type='TextMessage')], stop_reason='Maximum number of turns 3 reached.')

## Covered in Future Videos in the Module

# Aborting a Team

Different from stopping a team, aborting a team will immediately stop the team and raise a CancelledError exception.

In [25]:
from autogen_core import CancellationToken

cancellation_token = CancellationToken()

run2 = asyncio.create_task(
    Console(team.run_stream(task = 'Give a short Story about a lion atmost 40 words',cancellation_token=cancellation_token))
)

await asyncio.sleep(2)
cancellation_token.cancel()

try:
    result = await run2
except asyncio.CancelledError():
    print("Task Was Cancelled")

---------- TextMessage (user) ----------
Give a short Story about a lion atmost 40 words
---------- TextMessage (add_1_agent_first) ----------
Once, a brave lion saved a trapped cub from a hunter's net. Grateful, the animals crowned him as the jungle's protector. The lion, humbled by trust, vowed always to guard his kingdom with courage and kindness. His legacy lived on.
---------- TextMessage (add_1_agent_second) ----------
3


Error processing publish message for add_1_agent_third_400df8f6-9027-474f-abcf-9d1abd69f617/400df8f6-9027-474f-abcf-9d1abd69f617
Traceback (most recent call last):
  File "d:\Practice\AI\Autogen\autogen-env\Lib\site-packages\autogen_core\_single_threaded_agent_runtime.py", line 606, in _on_message
    return await agent.on_message(
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Practice\AI\Autogen\autogen-env\Lib\site-packages\autogen_core\_base_agent.py", line 119, in on_message
    return await self.on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Practice\AI\Autogen\autogen-env\Lib\site-packages\autogen_agentchat\teams\_group_chat\_sequential_routed_agent.py", line 67, in on_message_impl
    return await super().on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Practice\AI\Autogen\autogen-env\Lib\site-packages\autogen_core\_routed_agent.py", line 485, in on_message_impl
    return await h(self, 

TypeError: catching classes that do not inherit from BaseException is not allowed